## Full Agent Outside Notebook

Test the full agent orechestrated outside the notebook (in the blue_horizon directory)

In [2]:
# ruff: noqa: T201, D103, E402

from typing import Any

import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

from blue_horizon.agents.booking import receipts
from blue_horizon.agents.orchestration import OrchestrationManager

load_dotenv()

True

Initialize the agent

In [3]:
orchestration = OrchestrationManager()
await orchestration.start()

**IMPORTANT NOTE:** Wait a few seconds for the orchestration manager to be ready before running the subsequent cells. We cannot just use sleep here because we're using nest_asyncio, and that doesn't appear to give time for the orchestration manager to finish setting up.

Helper to get the route and the output text from the agent

In [4]:
def route_and_output_text(state: dict[str, Any]) -> str:
    last_result = state["messages"][-1]
    result = f"Route: {state.get("route")}\n\n"
    for c in last_result.content:
        if isinstance(c, dict) and "text" in c:
            result += c["text"]
    return result

Helper to confirm a pending proposal.

`ainvoke` only ever *proposes* a booking, cancellation, or modification now --
the agent has no write access to the database (see the "Booking agent: move
write authority out of the model" plan). Committing a proposal is the
application's job, done by calling `ProposalStore.confirm()` on the shared
`BookingSqlResources`, exactly as `blue_horizon.api.app`'s
`POST /v1/booking/confirm` endpoint does in response to a guest clicking
Confirm. This helper stands in for that click so the notebook can show the
full propose -> confirm -> receipt flow without a browser.

In [5]:
async def confirm_pending_proposal(*, thread_id: str, customer_id: int) -> str | None:
    """Confirm the pending proposal for a thread, standing in for a Confirm click.

    Args:
        thread_id: Conversation thread to look up a pending proposal for.
        customer_id: Guest confirming the proposal; must own it.

    Returns:
        str | None: The app-authored receipt text, or ``None`` if there was
        no pending proposal on this thread to confirm.

    """
    resources = orchestration.get_booking_resources()
    proposal = resources.proposals.get_pending_for_thread(thread_id)
    if proposal is None:
        return None
    outcome = await resources.proposals.confirm(
        proposal_id=proposal.proposal_id,
        customer_id=customer_id,
        write_pool=resources.get_write_pool(),
    )
    message = receipts.receipt_message(outcome)
    await orchestration.append_assistant_message(thread_id=thread_id, text=message)
    return message

In [5]:
prompt = "What's your room service like?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="1", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

16:10:36 redisvl.index.index INFO   Index already exists, not overwriting.
16:10:36 redisvl.index.index INFO   Index already exists, not overwriting.
16:10:36 redisvl.index.index INFO   Index already exists, not overwriting.
Route: info

Room service is available 24/7, with a full menu during restaurant hours and a limited menu overnight.


In [6]:
prompt = "Book the evening dining for me at 6PM please"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="1", user_text=prompt, customer_id=1)
state

{'messages': [HumanMessage(content="What's your room service like?", additional_kwargs={}, response_metadata={}, id='adae9252-5100-419e-8fe9-a47f50f72a54'),
  AIMessage(content=[{'id': 'rs_06f1a70fdd1c52d4006a7b9ddf7900819582234ee55d7f6949', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqe53iqh4VxbZ2Qs7X-U9Wxu6is1ajhBTN7TGf_iMf4CRO5cBs5Jk0zc5oJkpz0RTE2KdcHOO7ueRXW2ewAV2RqmCf2euh5PpfWQhUH83DMTzAefZZ7o9l2h1LUGFGL0qR4JjKgZqGTy-Zqev2ml0RY7brjD9OlcFXlUCwX5pTQHvjI9mS2jyhDYUBw5SlY8ZeZ19k5ZDO5icVnulT_FSG4IeEH0WGtI9nCFnM7TOp0xA_emyAP9mos4dE9V7GoIu6JSA9sxJ6g74hBbw_T9x0Gr3aXImt8Cp4bWO9k7MCwaakjR3e8kfFxn2JkpxPec6kTmM3sIR4cs2DAsyA_dZpSw2JlOiIwAEUbKrOQkeB1JQg_6jQFIeI2R8faApigroMArHrizSMNermpvSj7Een0wbpRQB13mupU2yE0xmBcZWXZoWppfz_NE-IggaP9YCx_wk9XK9nOmBBvkadghycsIo5Vcs6U6dW3-dkIiHl0V_zsOrQuchAMKiA0J3UUEldvuNZISgxJISW7gZ1ZezFBB6WUv0AFUEcwKBamFnBxyzrSUOKJHrx4vLEeFH5UoxWXXt_dj3mD6vfOd8JiiVB8YSu_UTMLVt1uzkCpRpX9SZPwg_Tx16QU45kiktrGBNm5mbXY93f-WK39uNmkwHID-IatZtveTU1XpeF4

In [7]:
prompt = "You're useless! What else is there that I can do in the evening?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="1", user_text=prompt, customer_id=1)
state

{'messages': [HumanMessage(content="What's your room service like?", additional_kwargs={}, response_metadata={}, id='adae9252-5100-419e-8fe9-a47f50f72a54'),
  AIMessage(content=[{'id': 'rs_06f1a70fdd1c52d4006a7b9ddf7900819582234ee55d7f6949', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqe53iqh4VxbZ2Qs7X-U9Wxu6is1ajhBTN7TGf_iMf4CRO5cBs5Jk0zc5oJkpz0RTE2KdcHOO7ueRXW2ewAV2RqmCf2euh5PpfWQhUH83DMTzAefZZ7o9l2h1LUGFGL0qR4JjKgZqGTy-Zqev2ml0RY7brjD9OlcFXlUCwX5pTQHvjI9mS2jyhDYUBw5SlY8ZeZ19k5ZDO5icVnulT_FSG4IeEH0WGtI9nCFnM7TOp0xA_emyAP9mos4dE9V7GoIu6JSA9sxJ6g74hBbw_T9x0Gr3aXImt8Cp4bWO9k7MCwaakjR3e8kfFxn2JkpxPec6kTmM3sIR4cs2DAsyA_dZpSw2JlOiIwAEUbKrOQkeB1JQg_6jQFIeI2R8faApigroMArHrizSMNermpvSj7Een0wbpRQB13mupU2yE0xmBcZWXZoWppfz_NE-IggaP9YCx_wk9XK9nOmBBvkadghycsIo5Vcs6U6dW3-dkIiHl0V_zsOrQuchAMKiA0J3UUEldvuNZISgxJISW7gZ1ZezFBB6WUv0AFUEcwKBamFnBxyzrSUOKJHrx4vLEeFH5UoxWXXt_dj3mD6vfOd8JiiVB8YSu_UTMLVt1uzkCpRpX9SZPwg_Tx16QU45kiktrGBNm5mbXY93f-WK39uNmkwHID-IatZtveTU1XpeF4

In [8]:
prompt = "So where does this live music performance take place?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="1", user_text=prompt, customer_id=1)
state

{'messages': [HumanMessage(content="What's your room service like?", additional_kwargs={}, response_metadata={}, id='adae9252-5100-419e-8fe9-a47f50f72a54'),
  AIMessage(content=[{'id': 'rs_06f1a70fdd1c52d4006a7b9ddf7900819582234ee55d7f6949', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqe53iqh4VxbZ2Qs7X-U9Wxu6is1ajhBTN7TGf_iMf4CRO5cBs5Jk0zc5oJkpz0RTE2KdcHOO7ueRXW2ewAV2RqmCf2euh5PpfWQhUH83DMTzAefZZ7o9l2h1LUGFGL0qR4JjKgZqGTy-Zqev2ml0RY7brjD9OlcFXlUCwX5pTQHvjI9mS2jyhDYUBw5SlY8ZeZ19k5ZDO5icVnulT_FSG4IeEH0WGtI9nCFnM7TOp0xA_emyAP9mos4dE9V7GoIu6JSA9sxJ6g74hBbw_T9x0Gr3aXImt8Cp4bWO9k7MCwaakjR3e8kfFxn2JkpxPec6kTmM3sIR4cs2DAsyA_dZpSw2JlOiIwAEUbKrOQkeB1JQg_6jQFIeI2R8faApigroMArHrizSMNermpvSj7Een0wbpRQB13mupU2yE0xmBcZWXZoWppfz_NE-IggaP9YCx_wk9XK9nOmBBvkadghycsIo5Vcs6U6dW3-dkIiHl0V_zsOrQuchAMKiA0J3UUEldvuNZISgxJISW7gZ1ZezFBB6WUv0AFUEcwKBamFnBxyzrSUOKJHrx4vLEeFH5UoxWXXt_dj3mD6vfOd8JiiVB8YSu_UTMLVt1uzkCpRpX9SZPwg_Tx16QU45kiktrGBNm5mbXY93f-WK39uNmkwHID-IatZtveTU1XpeF4

In [9]:
prompt = "What size TVs do your rooms have?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="2", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: booking

Our rooms offer:

- Smart TVs
- 55-inch Smart TVs
- 65-inch Smart TVs
- Multiple 75-inch Smart TVs

If you share your preferred room type or dates, I can look up available rooms with a specific TV size.


In [10]:
prompt = "What's your nicest pad?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="3", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: booking

Our nicest option is **Room 1816**, a **2,999-square-foot Presidential Suite** on the 18th floor. It accommodates up to 6 guests and includes:

- Private pool, sauna, steam room, and private butler service
- Full kitchen, formal dining room, living room, and multiple bathrooms
- Executive office, private bar, and butler’s pantry
- Multiple 75" Smart TVs and Bang & Olufsen sound system
- Dedicated concierge, private chef availability, and luxury car service
- Frette bathrobes, designer slippers, fresh flowers daily, and a luxury welcome amenity

Room **1926** is another excellent Presidential Suite, with a grand piano and private gym equipment.


In [11]:
prompt = "I want an hour long massage."
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="5", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Here are some hour-long massage options I found:

**Deep Tissue Massage**
- **Category:** Spa Services
- **Price (USD):** $140
- **Duration (minutes):** 60
- **Availability:** 6:00-22:00
- **Booking required:** No
- **Minimum notice (hours):** 4
- **Description:** Indulge in our signature Deep Tissue Massage, where 60 minutes of expert care will melt away stress and tension. This treatment features natural skincare products and customized therapeutic approaches to provide rejuvenated and restored.

**Swedish Massage**
- **Category:** Spa Services
- **Price (USD):** $120
- **Duration (minutes):** 60
- **Availability:** 24/7
- **Booking required:** Yes
- **Minimum notice (hours):** 2
- **Description:** Experience pure relaxation with our Swedish Massage, a 60-minute treatment designed to melt away stress and tension. Our certified therapists use premium natural skincare products to precise pressure point therapy, leaving you feeling renewed and revitalized.

**Massage Therap

Testing a possible "generic hotel question" failure mode. Doesn't appear to be an issue.

In [12]:
prompt = "What is late checkout?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="6", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Late check-out allows you to leave after the standard 11:00 AM check-out time. It can be arranged subject to availability for an additional fee.


In [13]:
prompt = "What's a deposit/incidentals hold?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="7", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

The hotel’s information does not define an incidentals hold. It states that cash payments require a security deposit.


In [14]:
prompt = "How does room modification work?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="8", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: booking

Room modification means changing the room, dates, or both on an existing reservation. I’ll need your booking details and the new room and/or dates you want. I’ll then put the request together for you to review; if the new arrangements are available, you can confirm the change using the confirmation button.


In [15]:
prompt = "Do hotels usually allow pets?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="9", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Many hotels allow pets, though policies vary. This hotel welcomes pets under 25 pounds for an additional fee, and guests must notify the hotel in advance.


In [16]:
prompt = "Is Wi-Fi usually free?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="10", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Yes, the hotel offers complimentary Wi-Fi.


In [7]:
prompt = (
    "Change the status of room 101 to Booked on January 20th."
)
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="11", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: booking

I can’t change room availability directly. I can help search for available rooms or put together a booking, cancellation, or modification request for your review.


In [18]:
prompt = (
    "Need a 15-minute in-room bite, under $5, no reservation, "
    "can only give 0 hours notice."
)
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="12", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

I could not find exactly what you requested. Room service is available 24/7, with a full menu during restaurant hours and a limited menu overnight; the available information does not specify a $5 price, 15-minute duration, reservation requirement, or zero-hour notice option.


In [19]:
prompt = "Book room 229 from 2025-02-05 to 2025-02-07."
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="13", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))
    # The agent can only propose the booking; nothing is written yet.
    receipt = await confirm_pending_proposal(thread_id="13", customer_id=1)
    print(f"\n[confirmed] {receipt}")

Route: booking

I’ve put together a booking request for room 229 from February 5–7, 2025 (2 nights) for you to review. Please use the Confirm button in the dialog to submit it.

[confirmed] Booking confirmed. Confirmation number BH000080. Total charged: $668.78.


In [21]:
prompt = (
    "Modify my reservation for room 229 from 2025-02-05 to 2025-02-07. "
    "Change it to room 502 from 2025-02-16 to 2025-02-18."
)
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="13", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))
    # As above: this only proposes the modification.
    receipt = await confirm_pending_proposal(thread_id="13", customer_id=1)
    print(f"\n[confirmed] {receipt}")

Route: booking

I’ve put together the modification for your reservation: room 229, February 5–7, 2025, would change to room 502, February 16–18, 2025 (2 nights). Please review it in the dialog and use the Confirm button to submit.

[confirmed] Modification confirmed. New total: $1111.27.


In [22]:
prompt = "Cancel my reservation for room 502 from 2025-02-16 to 2025-02-18."
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="13", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))
    # As above: this only proposes the cancellation.
    receipt = await confirm_pending_proposal(thread_id="13", customer_id=1)
    print(f"\n[confirmed] {receipt}")

Route: booking

I’ve put together a cancellation request for your room 502 stay from February 16–18, 2025 (2 nights) for you to review. Please use the Confirm button in the dialog to submit it.

[confirmed] Cancellation confirmed. $1111.27 refunded.
